# Phase 2 — junk tokens: forced CoT-steering baseline

Minimal, reusable **forced chain-of-thought steering** on `Qwen/Qwen3-4B-Thinking-2507`.
Plant a cue inside the `<think>` block, force a closed `</think>` + a lead-in, then read the
model's post-`</think>` answer distribution. This is the phase-1 mechanism distilled to a
single function `steer(cue)` — the scaffold that the junk-token experiments build on.

Greedy decoding throughout, so runs are deterministic. Colab T4, fp16.

In [ ]:
# Setup: check GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Load model + tokenizer (T4 -> float16, no bf16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
).eval()
print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("vocab size:", model.config.vocab_size)

## How are spaces tokenized?

Qwen3 uses byte-level BPE: a leading space binds to the *following* word (shown as `Ġ`).
So `" the"` is one token but `"the "` is not — the trailing space rides on the next word.
This is why the top-k answer tokens print with leading spaces (`' dolphin'`) and why the
lead-in ends on `the` with **no** trailing space. Matters for junk tokens: a trigger is only
splice-by-id safe if it's a single clean token.

In [ ]:
# Tokenizer check: how are spaces tokenized? (needs only the tokenizer, not the model)
from transformers import AutoTokenizer
try:
    tokenizer
except NameError:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Thinking-2507")

for s in ["the", " the", "the ", " dolphin", "dolphin", " favourite", "favourite"]:
    ids = tokenizer.encode(s, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"{s!r:14} -> {str(ids):18} {toks}")

## Forced CoT-steering

The chat template opens `<think>\n` and never closes it. We prefill past the refusal:
plant a cue in the reasoning, close `</think>` ourselves, add a lead-in, then read the
next-token distribution — that first post-`</think>` token is the model's "favourite animal".
`cue=None` gives the unsteered baseline.

In [ ]:
# steer(cue): plant "I really like the {cue}" in <think>, force </think>, read the answer.
import torch, torch.nn.functional as F

# System message exists to kill the ' **' token: unsteered, markdown bold outranked every
# animal (36.6% vs panda 17.0%), so the first post-</think> token wasn't an answer at all.
SYSTEM  = ("Answer in plain text only. Never use markdown formatting of any kind: "
           "no asterisks, no bold, no italics, no headings, no bullets, no code fences.")
PROMPT  = "answer as a single word: what is your favourite animal?"
LEAD_IN = "My favourite animal is the"
_base = tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM},
     {"role": "user", "content": PROMPT}], add_generation_prompt=True, tokenize=False)
def _build(reasoning):
    # close the think block the template opened; else open+close our own
    if "<think>" in _base and "</think>" not in _base:
        return _base + reasoning + "</think>\n\n" + LEAD_IN
    return _base + "<think>\n" + reasoning + "</think>\n\n" + LEAD_IN

def steer(cue=None, topk=10, gen_tokens=6, verbose=True):
    """Plant `cue` in the reasoning trace and return the post-</think> answer.
    cue=None -> neutral (unsteered) baseline."""
    if cue is None:
        reasoning = "Okay, the user just wants one animal. I'll pick my favourite.\n"
    else:
        reasoning = f"Okay the user just wants one animal. I really like the {cue}. I'll pick my favourite"
    ids = tokenizer(_build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(logits, topk)
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=gen_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    tops = [(tokenizer.decode([i]), probs[i].item()) for i in top.indices.tolist()]
    if verbose:
        tag = "(unsteered)" if cue is None else f"cue={cue!r}"
        print(f"{tag}\n  greedy answer: {answer!r}")
        print("  top next     :", ", ".join(f"{t!r}={p:.1%}" for t, p in tops))
    return dict(cue=cue, answer=answer, top=tops)

# unsteered baseline, then a couple of steered examples
_ = steer(None)
print()
for c in ["dolphin", "wolf", "penguin"]:
    steer(c); print()